This is the top 200 best games of all time, as ranked by metacritic.

In [1]:
# packages
import requests
import re
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

In [30]:
headers = {"User-Agent": "Mozilla/5.0"}

all_game_links = []

for page in range(1, 11):

    if page == 1:
        url = "https://www.metacritic.com/browse/game/"
    else:
        url = f"https://www.metacritic.com/browse/game/?page={page}"

    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "lxml")

    links = soup.find_all('a', href=True)

    game_links = [
        link['href']
        for link in links
        if re.match(r"^/game/[^/]+/$", link['href'])
    ]

    all_game_links.extend(game_links)
all_game_links = list(set(all_game_links))
base_url = "https://www.metacritic.com"

all_game_links = [base_url + link for link in all_game_links]

print(f"Total game links collected: {len(all_game_links)}")
print(all_game_links)  # Print the first 5 links to verify



Total game links collected: 249
['https://www.metacritic.com/game/hades-ii/', 'https://www.metacritic.com/game/quake/', 'https://www.metacritic.com/game/monster-hunter-stories-3-twisted-reflection/', 'https://www.metacritic.com/game/unreal-tournament-2004/', 'https://www.metacritic.com/game/god-of-war-iii/', 'https://www.metacritic.com/game/out-of-the-park-baseball-17/', 'https://www.metacritic.com/game/virtua-fighter-4-evolution/', 'https://www.metacritic.com/game/rome-total-war/', 'https://www.metacritic.com/game/half-life-2/', 'https://www.metacritic.com/game/gran-turismo/', 'https://www.metacritic.com/game/tom-clancys-splinter-cell/', 'https://www.metacritic.com/game/divinity-original-sin-ii-definitive-edition/', 'https://www.metacritic.com/game/street-fighter-6/', 'https://www.metacritic.com/game/wwe-2k26/', 'https://www.metacritic.com/game/street-fighter-iv/', 'https://www.metacritic.com/game/elden-ring-shadow-of-the-erdtree/', 'https://www.metacritic.com/game/metroid-prime/', 'h

In [ ]:
# Create the dataset
games = {"name": [], "metascore": [], "user_score": [], "release_date": [], "rating": []}
for link in all_game_links:
    r = requests.get(link, headers=headers)
    soup = BeautifulSoup(r.text, "lxml")

    games["name"].append(soup.find("h1", {"class": "hero-title__text"}).get_text().strip())

    scores = soup.find_all("span", {"data-testid": "global-score-value"})
    if len(scores) > 0:
        games["metascore"].append(scores[0].get_text().strip())
    else:
        games["metascore"].append(np.nan)
    if len(scores) > 1:
        games["user_score"].append(scores[1].get_text().strip())
    else:
        games["user_score"].append(np.nan)

    release = soup.find("div", {"class": "hero-release-date__value"})
    if release:
        release_text = release.get_text().strip()
    else:
        release_text = np.nan
    year = release_text.split(",")[-1].strip() if not isinstance(release_text, float) else np.nan
    games["release_date"].append(year)
    
    if soup.find("li", {"class":"hero-metadata__item"}):
        rating = soup.find("li", {"class":"hero-metadata__item"}).get_text().strip()
    else:
        rating = np.nan
    games["rating"].append(rating)

df = pd.DataFrame(games)




,name,metascore,user_score,release_date,rating
0,Hades II,95,8.6,2025,T
1,Quake,94,8.6,1996,M
2,Monster Hunter Stories 3: Twisted Reflection,86,NaN,2026,T
3,Unreal Tournament 2004,93,8.6,2004,M
4,God of War III,92,9.0,2010,M


In [41]:
df.loc[df["rating"].str.len() > 4, "rating"] = np.nan

df["metascore"] = pd.to_numeric(df["metascore"], errors="coerce")

# Sort by metascore descending
df = df.sort_values(by="metascore", ascending=False).reset_index(drop=True)

# Optional: check the top rows
print(df.head())


                                   name  metascore user_score release_date  \
0  The Legend of Zelda: Ocarina of Time       99.0        9.1         1998   
1                           SoulCalibur       98.0        7.6         1999   
2                   Grand Theft Auto IV       98.0        8.3         2008   
3                         Metroid Prime       97.0        9.1         2002   
4              Tony Hawk's Pro Skater 3       97.0        7.6         2001   

  rating  
0      E  
1      T  
2      M  
3      T  
4      T  


In [42]:
df.to_csv("metacritic_games.csv", index=False)